In [1]:
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import optuna


warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
# DATA = '../data/'
DATA = '/kaggle/input/intern-task-regression-courier-deficit-challenge/'
SEED = 42

In [2]:
# Загрузка
train = pd.read_csv(DATA + "train.csv", parse_dates=['calendar_dt'])
test = pd.read_csv(DATA + "test.csv")
facts = pd.read_csv(DATA + "facts.csv", parse_dates=['calendar_dt'])
shifts = pd.read_csv(DATA + "shifts_prediction.csv", parse_dates=['calendar_dt'])

# Просмотр размерности датасетов
for name, data in [("train", train), ("test", test), ("facts", facts), ("shifts", shifts)]:
    print(f"{name}: {data.shape}")

train: (8220, 3)
test: (2438, 1)
facts: (10660, 13)
shifts: (223470, 5)


In [3]:
train['store_id'].value_counts()

store_id
60548799-a410-11eb-85a9-1c34dae33151    95
12ff0515-9aa3-11ea-bc7d-0050560306e1    94
cf69cefd-2016-11eb-8599-1c34dae33151    94
f46701f1-af2c-11ea-b969-0050560306e1    94
4c5db80c-3fe0-11ed-885a-08c0eb32014b    87
                                        ..
75843b61-71bc-11ed-b96e-08c0eb32008b     2
41722ab1-955a-11f0-aa3c-be3af2b6059f     1
570cadce-8a43-11f0-a97c-be3af2b6059f     1
bf446169-9d1e-11f0-a5cb-7cc255b74513     1
1cbeffb6-f4e7-11ef-aa23-be3af2b6059f     1
Name: count, Length: 2453, dtype: int64

In [4]:
# Смотрю сколько магазинов есть на каждой дате
store = train['calendar_dt'].value_counts().sort_index()
print(store.tail())
print(f"Тотал по датам: {len(store)}; дат, где больше 1000 сторов: {(store > 1000).sum()}")
# Порог на проверку пробовал от 100 до 1000, результат не меняется -> есть 3 недели, когда есть данные по большей части сторов

calendar_dt
2025-10-20      13
2025-10-27      13
2025-11-03    2431
2025-11-10    2436
2025-11-17    2439
Name: count, dtype: int64
Тотал по датам: 98; дат, где больше 1000 сторов: 3


Почти у всех магазинов есть только 3 недели истории.
3 недели мало, чтобы предсказывать по прошлому.
Поэтому буду опираться на то, что известно в начале недели: факты за прошлую неделю и план на текущую

In [5]:
# Склейка датасетов
# добавим в test дату, так как по условию нужно делать предикт на 24 ноября
TEST_DT = pd.Timestamp('2025-11-24')
test['calendar_dt'] = TEST_DT

# складываем сеты
df = pd.concat([train, test], ignore_index=True)

# тяну признаки по дате и id, тк один стор встречается на разных датах и одна дата может быть у разных сторов
df = df.merge(facts, on=['calendar_dt', 'store_id'], how='left')
df = df.merge(shifts, on=['calendar_dt', 'store_id'], how='left')
# сортирую, чтобы все недели одного стора шли подряд
df = df.sort_values(['store_id', 'calendar_dt']).reset_index(drop=True)
print(df.duplicated().sum())
print(df.shape)

0
(10658, 17)


In [6]:
# добавляю бинарную колонку, чтобы видеть где train и test
df['is_test'] = df['calendar_dt'] == TEST_DT
# проверка пропусков
df[df.is_test].isna().sum()

calendar_dt                           0
store_id                              0
target                             2438
fact_staff_value_lag_1                0
fact_load_factor_lag_1                6
num_available_couriers_lag_1          0
fact_num_orders_lag_1                 0
fact_percent_lateness_lag_1           0
city_nm                               0
store_lifetime_in_days                0
fact_staff_churn                      0
flag_high_load_lag_1                  0
marketing_costs_lag_1               719
fact_couriers_with_shifts_lag_1       6
predicted_staff_value                 0
predicted_num_orders                  0
predicted_load_factor                 0
is_test                               0
dtype: int64

In [7]:
df.head()

,calendar_dt,store_id,target,fact_staff_value_lag_1,fact_load_factor_lag_1,num_available_couriers_lag_1,fact_num_orders_lag_1,fact_percent_lateness_lag_1,city_nm,store_lifetime_in_days,fact_staff_churn,flag_high_load_lag_1,marketing_costs_lag_1,fact_couriers_with_shifts_lag_1,predicted_staff_value,predicted_num_orders,predicted_load_factor,is_test
0,2025-11-03,000fade4-e8dc-11ed-b10a-08c0eb31fffb,1.0,1,0.526316,10,10,NaN,Ульяновск,888.0,1.0,1,3.683577e+08,19.0,7.0,260.0,4.96,False
1,2025-11-10,000fade4-e8dc-11ed-b10a-08c0eb31fffb,1.0,8,1.941176,13,33,69.565217,Ульяновск,895.0,1.0,1,NaN,17.0,6.0,290.0,5.06,False
2,2025-11-17,000fade4-e8dc-11ed-b10a-08c0eb31fffb,4.0,8,2.400000,12,36,66.666667,Ульяновск,902.0,4.0,1,4.254502e+10,15.0,7.0,280.0,4.86,False
3,2025-11-24,000fade4-e8dc-11ed-b10a-08c0eb31fffb,NaN,11,2.294118,12,39,94.444444,Ульяновск,909.0,2.0,1,NaN,17.0,8.0,320.0,4.66,True
4,2025-11-03,0022f1b0-b8f8-11ee-b10b-08c0eb31fffb,1.0,1,0.833333,10,10,NaN,Набережные Челны,617.0,1.0,1,NaN,12.0,5.0,220.0,6.17,False


In [8]:
by_date = train.groupby('calendar_dt')['target'].agg(['count', 'mean'])
by_date

,count,mean
calendar_dt,,
2024-01-01,4,11.250000
2024-01-08,4,15.000000
2024-01-15,4,15.000000
2024-01-22,4,12.750000
2024-01-29,4,13.500000
...,...,...
2025-10-20,13,2.307692
2025-10-27,13,1.153846
2025-11-03,2431,1.973673


In [9]:
by_date.groupby(by_date.index.year)['mean'].agg(['min', 'max'])

,min,max
calendar_dt,,
2024,0.777778,15.000000
2025,0.416667,4.454545


До ноября 25 года малое количество сторов записано + большие скачки среднего по таргету за 24 и 25 годы
Думаю логичнее будет использовать данные за ноябрь, как минимум из-за количества известных сторов, так как
за старые даты 24 года по 4-13 сторов не стоит делать вывод про всю сеть

In [10]:
full_week = df[(df['calendar_dt'] >= '2025-11-03') & (~df['is_test'])]
full_week.shape

(7306, 18)

In [11]:
2431 + 2436 + 2439

7306

Теперь посмотрю на корреляцию признаков с таргетом

In [12]:
full_week.select_dtypes('number').corr()['target'].sort_values()

predicted_load_factor             -0.438711
marketing_costs_lag_1             -0.013756
num_available_couriers_lag_1       0.019596
fact_load_factor_lag_1             0.020431
store_lifetime_in_days             0.021940
predicted_num_orders               0.073593
fact_num_orders_lag_1              0.181774
fact_couriers_with_shifts_lag_1    0.216714
fact_percent_lateness_lag_1        0.239168
fact_staff_value_lag_1             0.336015
predicted_staff_value              0.346320
flag_high_load_lag_1               0.616515
fact_staff_churn                   0.886640
target                             1.000000
Name: target, dtype: float64

Сильнее всего с дефицитом связано число уволившихся курьеров (0.89), потом повышенный спрос (0.62) и нагрузка на курьера (-0.44),
тут логично, что чем больше курьеров будут выполнять заказов в час, тем меньше нужно будет людей на точку

На основе корреляции можно выдвинуть гипотезу, что наш target приблизительно равен уволенным, что в целом по бизнес логике сходится (нам нужно нанять такое же количество людей, чтобы не допустить дефицит кадров на точке). Тогда можно ввести разницу между target и churn, тогда она должна выходить в 0

In [13]:
df['difference'] = df['target'] - df['fact_staff_churn']
full = df[(df['calendar_dt'] >= '2025-11-03') & (~df['is_test'])]
full[['target', 'fact_staff_churn', 'difference', 'flag_high_load_lag_1']].head()

,target,fact_staff_churn,difference,flag_high_load_lag_1
0,1.0,1.0,0.0,1
1,1.0,1.0,0.0,1
2,4.0,4.0,0.0,1
4,1.0,1.0,0.0,1
5,1.0,1.0,0.0,1


In [14]:
print((full['difference'] == 0).mean())
print(full['difference'].min())
print(full['difference'].describe())

0.7199562003832466
0.0
count    7306.000000
mean        0.539693
std         1.312343
min         0.000000
25%         0.000000
50%         0.000000
75%         1.000000
max        26.000000
Name: difference, dtype: float64


difference = target − churn равна 0 в 72% случаев и никогда не бывает отрицательной. Тогда, дефицит = уволившиеся + n (в каких-то случаях).

In [15]:
pd.crosstab(full['difference'], full['flag_high_load_lag_1'])

flag_high_load_lag_1,0,1
difference,,
0.0,2304,2956
1.0,0,1193
2.0,0,532
3.0,0,97
4.0,0,61
5.0,0,54
6.0,0,39
7.0,0,26
8.0,0,14


Получается, что при обычном спросе difference всегда 0 и за доп. штатом идем при повышенной нагрузке.
Дефицит = число уволившихся (при обычном спросе) => модель нужна только для предсказания difference у магазинов с повышенным спросом

Нужно понять от чего зависит difference при загруженности 

In [16]:
high_load = full[full['flag_high_load_lag_1'] == 1]
high_load.shape

(5002, 19)

In [17]:
high_load['difference'].value_counts().sort_index()

difference
0.0     2956
1.0     1193
2.0      532
3.0       97
4.0       61
5.0       54
6.0       39
7.0       26
8.0       14
9.0       10
10.0       4
11.0       5
12.0       4
13.0       3
14.0       1
19.0       1
22.0       1
26.0       1
Name: count, dtype: int64

Чаще всего 0, потом 1 и 2, а дальше значения быстро редеют

In [18]:
high_load.select_dtypes('number').drop(columns=['target', 'flag_high_load_lag_1']).corr()['difference'].sort_values()

predicted_load_factor             -0.216322
marketing_costs_lag_1             -0.009175
store_lifetime_in_days             0.011954
fact_load_factor_lag_1             0.053492
fact_percent_lateness_lag_1        0.098489
num_available_couriers_lag_1       0.131875
predicted_num_orders               0.151865
fact_couriers_with_shifts_lag_1    0.175828
fact_num_orders_lag_1              0.189283
fact_staff_value_lag_1             0.260567
fact_staff_churn                   0.264865
predicted_staff_value              0.283513
difference                         1.000000
Name: difference, dtype: float64

сильных корреляций не видно, скорее всего какие то комбинации признаков могут дать хорошую связь + нужно не забывать про города, так как там думаю будет видна разница по нагрузке => difference

In [19]:
high_load.groupby('city_nm')['difference'].agg(['mean', 'median', 'count']).sort_values('count', ascending=False).head(10)

,mean,median,count
city_nm,,,
Москва,0.706693,0.0,1016
Санкт-Петербург,1.112561,0.0,613
Новосибирск,1.295567,1.0,203
Екатеринбург,0.904459,0.0,157
Краснодар,0.531469,0.0,143
Воронеж,0.703125,0.0,128
Омск,0.754237,0.0,118
Самара,0.649573,0.0,117
Тюмень,0.474138,0.0,116


Перейдем к доп признакам:
1) первая идея это смотреть на наличие курьеров (сколько мы предиктим и сколько есть заявок)
2) из первой фичи можно реализовать так же кол-во, которое реально вышло (предикт - факт)
3) думаю стоит смотреть на соотношение предикта тотал заказов и фактическое количество
4) у нас есть информация по заявкам курьеров о выходе на смену и их фактическое количество (собираем соотношение)
5) переведу название городов в категории

In [20]:
df['deficit_staff'] = df['predicted_staff_value'] - df['num_available_couriers_lag_1']
df['real_staff'] = df['predicted_staff_value'] - df['fact_couriers_with_shifts_lag_1']
df['ratio_orders'] = df['predicted_num_orders'] / df['fact_num_orders_lag_1']
df['ratio_couriers'] = df['fact_couriers_with_shifts_lag_1'] / df['num_available_couriers_lag_1']
df['city_nm'] = df['city_nm'].astype('category')

# есть деление и риск получить бесконечность
df = df.replace([np.inf, -np.inf], np.nan)

In [21]:
FEATURES = [
    'predicted_staff_value', 'predicted_num_orders', 'predicted_load_factor',
    'fact_staff_value_lag_1', 'fact_load_factor_lag_1', 'num_available_couriers_lag_1',
    'fact_num_orders_lag_1', 'fact_percent_lateness_lag_1', 'fact_couriers_with_shifts_lag_1',
    'fact_staff_churn', 'marketing_costs_lag_1',
    'store_lifetime_in_days', 'city_nm',
    'deficit_staff', 'real_staff', 'ratio_orders', 'ratio_couriers',
]

In [22]:
def wape(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return np.abs(y_true - y_pred).sum() / y_true.sum()


def make_prediction(data, difference_pred):
    difference_pred = np.clip(difference_pred, 0, None)
    difference_pred = np.round(difference_pred)
    flag = data['flag_high_load_lag_1'].values
    churn = data['fact_staff_churn'].values
    return churn + flag * difference_pred


In [23]:
# проверка
# учимся на прошлом, а проверяем на следующей неделе
# бейзлайн это churn

VALID_DATES = ['2025-11-10', '2025-11-17']
labeled = df[~df['is_test']]

for valid_date in VALID_DATES:
    valid = labeled[labeled['calendar_dt'] == valid_date]

    pred = make_prediction(valid, 0)
    print(valid_date, 'бейзлайн (только churn):', round(wape(valid['target'], pred), 4))

2025-11-10 бейзлайн (только churn): 0.2287
2025-11-17 бейзлайн (только churn): 0.2385


In [24]:
# данные до проверочной недели + повышенная загруженность, когда нужна добавка людей
# для каждой проверочной недели своя обучающая часть, поэтому считаем внутри цикла
for valid_date in VALID_DATES:
    train_part = labeled[(labeled['calendar_dt'] < valid_date) & (labeled['flag_high_load_lag_1'] == 1)]
    print(valid_date, 'строк для обучения:', len(train_part))

2025-11-10 строк для обучения: 1997
2025-11-17 строк для обучения: 3780


1) wape считает по модулю, а их минимизирует медиана, когда l1 это и делает
2) lr=0.03 и n_estimators=400: маленький шаг делает обучение стабильнее, а чтобы модель успела доучиться,
   деревьев нужно больше
3) num_leaves=15: модель учится всего на 2к–4к строк. Дерево с большим числом листьев начнёт запоминать отдельные магазины вместо    общих закономерностей
4) min_child_samples=30: в каждый лист должно попасть минимум 30 строк. В EDA видно,
   что большие значения difference редкие
   без этого ограничения модель могла бы построить отдельное правило под несколько таких магазинов

In [25]:
PARAMS = dict(
    objective='l1',
    n_estimators=400,
    learning_rate=0.03,
    num_leaves=15,
    min_child_samples=30,
    random_state=42,
    verbose=-1
)

In [26]:
for valid_date in VALID_DATES:
    train_part = labeled[(labeled['calendar_dt'] < valid_date) & (labeled['flag_high_load_lag_1'] == 1)]
    valid = labeled[labeled['calendar_dt'] == valid_date]

    model = lgb.LGBMRegressor(**PARAMS)
    model.fit(train_part[FEATURES], train_part['difference'])

    difference_pred = model.predict(valid[FEATURES])
    pred = make_prediction(valid, difference_pred)

    print(
        valid_date, 'бейзлайн:', round(wape(valid['target'], make_prediction(valid, 0)), 4),
        '/// model:', round(wape(valid['target'], pred), 4),
    )

2025-11-10 бейзлайн: 0.2287 /// model: 0.1748
2025-11-17 бейзлайн: 0.2385 /// model: 0.1706


результат неплохой, но ради интереса хочется воспользоваться еще optuna, возможно, это избыточно

In [27]:
def score(params):
    scores = []
    for valid_date in VALID_DATES:
        train_part = labeled[(labeled['calendar_dt'] < valid_date) & (labeled['flag_high_load_lag_1'] == 1)]
        valid = labeled[labeled['calendar_dt'] == valid_date]

        model = lgb.LGBMRegressor(**params)
        model.fit(train_part[FEATURES], train_part['difference'])
        pred = make_prediction(valid, model.predict(valid[FEATURES]))

        scores.append(wape(valid['target'], pred))
    return np.mean(scores)

# optuna
def objective(trial):
    params = dict(
        objective='l1',
        random_state=42,
        verbose=-1,
        subsample_freq=1,
        n_estimators=trial.suggest_int('n_estimators', 100, 1000, step=50),
        learning_rate=trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        num_leaves=trial.suggest_int('num_leaves', 4, 63),
        min_child_samples=trial.suggest_int('min_child_samples', 10, 200, log=True),
        subsample=trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree=trial.suggest_float('colsample_bytree', 0.5, 1.0),
        reg_lambda=trial.suggest_float('reg_lambda', 1e-3, 10, log=True),
    )
    return score(params)



In [28]:
optuna.logging.set_verbosity(optuna.logging.WARNING)

study = optuna.create_study(direction='minimize', sampler=optuna.samplers.TPESampler(seed=42))

study.enqueue_trial({
    'n_estimators': 400, 'learning_rate': 0.03, 'num_leaves': 15, 'min_child_samples': 30,
    'subsample': 1.0, 'colsample_bytree': 1.0, 'reg_lambda': 0.001,
})

study.optimize(objective, n_trials=50)

print('лучший WAPE:', round(study.best_value, 4))
print(study.best_params)

лучший WAPE: 0.1702
{'n_estimators': 350, 'learning_rate': 0.01361062258494573, 'num_leaves': 24, 'min_child_samples': 41, 'subsample': 0.6984727917423988, 'colsample_bytree': 0.7393101891645488, 'reg_lambda': 0.5802018860203916}


улучшение небольшое (0.1727(avg за 2 недели) -> 0.1702), а проверочных недель всего две, поэтому выигрыш может быть случайным


строим финальную версию

In [29]:
BEST_PARAMS = dict(objective='l1', random_state=SEED, verbose=-1, subsample_freq=1, **study.best_params)
BEST_PARAMS

{'objective': 'l1',
 'random_state': 42,
 'verbose': -1,
 'subsample_freq': 1,
 'n_estimators': 350,
 'learning_rate': 0.01361062258494573,
 'num_leaves': 24,
 'min_child_samples': 41,
 'subsample': 0.6984727917423988,
 'colsample_bytree': 0.7393101891645488,
 'reg_lambda': 0.5802018860203916}

In [30]:
# все размеченные строки с повышенным спросом
train_all = labeled[labeled['flag_high_load_lag_1'] == 1]
test_part = df[df['is_test']].copy()

final_model = lgb.LGBMRegressor(**BEST_PARAMS)
final_model.fit(train_all[FEATURES], train_all['difference'])

,boosting_type,'gbdt'
,num_leaves,24
,max_depth,-1
,learning_rate,0.01361062258494573
,n_estimators,350
,subsample_for_bin,200000
,objective,'l1'
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,41


In [31]:
difference_pred = final_model.predict(test_part[FEATURES])
test_part['target'] = make_prediction(test_part, difference_pred).astype(int)

In [32]:
submission = (
    test[['store_id']]
    .merge(test_part[['store_id', 'target']], on='store_id', how='left')
    .sort_values('store_id')
)

sample_sub = pd.read_csv(DATA + 'sample_submission.csv')
assert len(submission) == len(sample_sub)
assert submission['target'].notna().all()
assert (submission['store_id'].values ==
        sample_sub.sort_values('store_id')['store_id'].values).all()

submission['target'].describe()

count    2438.000000
mean        2.377358
std         2.469081
min         0.000000
25%         0.000000
50%         2.000000
75%         4.000000
max        11.000000
Name: target, dtype: float64

In [33]:
submission.to_csv('../submission.csv', index=False)
pd.read_csv('../submission.csv').head()

,store_id,target
0,000fade4-e8dc-11ed-b10a-08c0eb31fffb,2
1,0022f1b0-b8f8-11ee-b10b-08c0eb31fffb,0
2,00440ac1-6a1d-11eb-85a3-1c34dae33151,1
3,00442959-9671-11ec-ae6d-08c0eb320147,8
4,00562194-569a-11ec-a0ee-ec0d9a21b021,4


Итоги

Главная находка в таргете: дефицит = число уволившихся + добавка. При обычном спросе добавка всегда 0, поэтому для примерно трети магазинов ответ известен точно. Модель нужна только для того, чтобы предсказать добавку у магазинов с повышенным спросом.

Результаты на проверке (среднее по двум неделям): бейзлайн "дефицит = число уволившихся" дал WAPE 0.2336, LightGBM с L1-лоссом 0.1727, после подбора параметров через Optuna 0.1702. Основной прирост дали постановка задачи и модель с L1-лоссом, Optuna добавила немного.

Ограничения: формула таргета проверена только на 3 неделях, и если она перестанет выполняться, модель будет ошибаться. Проверочных недель всего две, поэтому оценка качества не очень надёжная.

Мои предложения по улучшению:

1. Добавить признаки по праздникам, так как в праздничные недели спрос, скорее всего, выше. Данные недельные, поэтому признак может быть таким: число праздничных дней на неделе или флаг "неделя перед праздником". Сейчас из этого много не получить, потому что в обучении только 3 недели и праздничная среди них одна (3-4 ноября), но с накоплением истории признак станет полезным. Также можно добавить погоду по городам (в дождь, ураган, шторм и тд.. не все люди самостоятельно пойдут в магазины).

2. Добавить алерты, чтобы уже работать в рамках какой-то системы, а не одной модели. Каждую неделю, когда становится известен реальный дефицит, считать WAPE и поднимать алерт, если ошибка заметно выросла по сравнению с проверкой (около 0.17) или приблизилась к бейзлайну (около 0.23), если перестало выполняться правило "при обычном спросе дефицит = уволившиеся" или если сильно изменились входные данные. В этом случае переобучить модель и перепроверить формулу таргета.

3. Попробовать другие бустинговые фреймворки (например, catboost)


*late submission на Kaggle модель получила WAPE 0.179*
